**`¿qué atributos describen una fase?`**    
el modelo va aprender sobre los atributos de la fase.  

**`¿cómo se agrupan esas fases dentro de familias?`**     
no he de trabajar con fase - máquina sino más bien familias de fases - máquinas.   

**`revisar los tiempos reales`**      
puede haber mucho ruido y he de controlarlo.   

**`construir la tabla de conteo`**
conteo de familias de fases x máquinas

atributos → familias → tabla de conteos familia × máquina → lectura de dónde hay decisión

In [3]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
ruta_FASE = os.path.join("res\\raw", "CPRMOTask.csv")
print(ruta_FASE) 

In [ ]:
columns_FASE = ["IDMOTask", "CodCompany", "IDManufacturingOrder", "CodMOTask", "Description", "Notes", "Order", "PreparationTime", "ExecutionTime", "RepetitionTime", "TimeUnit", "ProductEntry", "MaterialExit", "DelimitationType", "DelimitationDate", "AllowFraccionate", "MinimumFraction", "IDUsualTask", "TaskType", "IDMOTaskParent", "RepetitionLot", "External", "IDArticleExternal", "Quantity", "IDSupplier", "IDTaskType", "RepetitionTimeReal", "PlannedStartDate", "PlannedEndDate", "Group", "IDTask", "PercentProgress", "SeriesRequired", "QualityStatus", "InspectedDate", "Priority", "OptimizationFeature", "Fixed", "WithoutTimetable", "OccupancyType", "OccupancyHours", "OccupancyPercent", "ManualEndDate", "ManualStartDate", "Blocked", "DelimitationObligatory", "IDBudgetCategory", "ResourcePlanType", "SplitNumber", "ManualPlanning", "IDBudgetEmployee", "IDBudgetMachine", "IDBudgetTool", "IDBudgetMaterial", "IDBudgetOtherCost", "RowTimestamp", "WorkUnits", "LastModifiedBy", "IDProductFormula", "DateReplanify", "LotsOrder", "PrintLabels", "AutoExecutionImputation", "IDSiteMaterial", "IDWarehouseMaterial", "GroupingNumber", "CreationTimestamp", "RowCreatedBy", "BlockedMaterial", "BlockedCost", "ImputationCostUpdated", "Canceled", "RealEndDate", "IDCheckList", "NotOverlapCode", "TiedCode", "GroupedManufacturing", "ReleaseKeptResources", "DelayMargin", "BlockedChangeMaterial", "IDHourType", "Color", "SetToQtyConversionFactor", "AdjustTaskToDay", "PercentProgressType", "StatusTaskMES"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)          

df_FASE = pd.read_csv(ruta_FASE, sep=";", header=None, names=columns_FASE)


In [ ]:
display(df_FASE.head(1))

In [ ]:
display(df_FASE.shape)

## Limpiar fases Externas o Canceladas
Todas estas fases no sirven para entrenar al modelo.

In [ ]:
def filtrar_fases_no_validas(df):
    """Conserva solo las fases que NO son externas, NI canceladas.
    Descarta cualquier instancia con un 1 en External o Canceled.
    Devuelve un DataFrame nuevo (no modifica el original)."""

    n_inicial = df.shape[0]

    # Conservar filas donde las tres valen 0 (o NULL tratado como 0).
    # fillna(0): si hubiera NULLs, los tratamos como "no marcado" = válido.
    mask_validas = (
        (df['External'].fillna(0) == 0) &
        (df['Canceled'].fillna(0) == 0)
    )

    df_out = df[mask_validas].copy()

    # Desglose de cuántas se van por cada motivo (informativo; pueden solaparse)
    n_ext = (df['External'].fillna(0) == 1).sum()
    n_can = (df['Canceled'].fillna(0) == 1).sum()

    print(f"Partida: {n_inicial} fases")
    print(f"  Externas:  {n_ext}") # External=1
    print(f"  Canceladas: {n_can}") # Canceled=1
    print(f"Conservadas: {df_out.shape[0]} fases "
          f"(descartadas {n_inicial - df_out.shape[0]})")

    return df_out

In [ ]:
df_FASE = filtrar_fases_no_validas(df_FASE)

Puede ser que haya fases Externas infiltradas, voy hacer uns inspección en base al nombre de la fase

In [ ]:
posibles_externas = "ext|subcon"

df_infiltradas = df_FASE[df_FASE["Description"].str.contains(posibles_externas, case=False, na=False)]

print(f"Número de fases externas infiltradas (no marcadas como externas siendolo): {df_infiltradas.shape[0]}" )
#display(df_infiltradas.sample(df_infiltradas.shape[0]))

Estas fases externas infiltradas no las quiero, asi que las elimino del dataset con el que trabajaré.

In [ ]:
print(f"Fases totales de eliminar infiltradas antes: {df_FASE.shape[0]}")

df_FASE = df_FASE[~df_FASE["Description"].str.contains(posibles_externas, case=False, na=False)]

print(f"Se eliminarán: {df_infiltradas.shape[0]}")
print(f"Fases totales después de eliminar infiltradas: {df_FASE.shape[0]}")

## Limpiar fases sin tiempos teóricos
Todas las fases que no tengan tiempos en los atributos `PreparationTime`  o `ExecutionTime` las elimino del dataset, esas instancias no nos serán útiles.

In [ ]:
print(f"Instancias totales: {df_FASE.shape[0]}")
df_FASE_con_TIEMPOS = df_FASE[(df_FASE["PreparationTime"] != 0) | (df_FASE["ExecutionTime"] != 0)]

print(f"Número de instancias con tiempos teóricos: {df_FASE_con_TIEMPOS.shape[0]}")
print(f"Número de instancias sin tiempo teórico (descartar): {df_FASE.shape[0] - df_FASE_con_TIEMPOS.shape[0]}")

## Limpieza por fecha
Vamos a trabajar con instancias solo a partir del Septiembre 2025

In [ ]:
def filtrar_desde_fecha(df, fecha_desde, columna='CreationTimestamp'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea >= fecha_desde.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro >= {fecha_desde}.")

    limite = pd.to_datetime(fecha_desde)
    df_out = df_out[df_out[columna] >= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (desde {fecha_desde}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [ ]:
df_FASE_valid = filtrar_desde_fecha(df_FASE_con_TIEMPOS, fecha_desde='2025-06-01')
print(df_FASE_valid.shape)

## Limpieza de atributos 

Tenemos 86 columnas, vamos hacer la selección de atributos:    

Atributos que sobran: [CodCompany, CodMOTask, Notes, Order, RepetitionTime, DelimitationType, DelimitationDate, AllowFraccionate, MinimumFraction, RepetitionLot, IDArticleExternal, RepetitionTimeReal, Group, SeriesRequired, QualityStatus, InspectedDate, Priority, OptimizationFeature, Fixed, WithoutTimetable, OccupancyType, DelimitationObligatory, IDBudgetCategory, ResourcePlanType, SplitNumber, ManualPlanning, IDBudgetEmployee, IDBudgetMachine, IDBudgetTool, IDBudgetMaterial, IDBudgetOtherCost, RowTimestamp, LastModifiedBy, WorkUnits, IDProductFormula, LotsOrder, PrintLabels, AutoExecutionImputation, IDSiteMaterial, IDWarehouseMaterial, GroupingNumber, CreationTimestamp, RowCreatedBy, BlockedMaterial, BlockedCost, ImputationCostUpdated, IDCheckList, NotOverlapCode, TiedCode, GroupedManufacturing, ReleaseKeptResources, DelayMargin, BlockedChangeMaterial, IDHourType, Color, SetToQtyConversionFactor, AdjustTaskToDay, PercentProgressType, StatusTaskMES, Blocked]     

Atributos que necesito saber que posibles valores hay, y si son todos iguales, puedo prescindir de estos para el df: [WorkUnits, TimeUnit, OccupancyHours, OccupancyPercent]    

In [ ]:
def inspeccionar_candidatos(df):
    """Inspecciona columnas candidatas a ser prescindibles.
    Para cada una muestra cuántos valores distintos tiene y su distribución.
    Marca cuáles son constantes (un solo valor) y por tanto prescindibles.
    NO se elimina nada, solo carácter informativo."""

    columnas = ['TimeUnit', 'OccupancyHours', 'OccupancyPercent']

    constantes = []
    con_varianza = []

    for col in columnas:
        if col not in df.columns:
            print(f"[{col}] no está en el DataFrame\n")
            continue

        n = df[col].nunique(dropna=False)
        print(f"====== {col} ======")
        print(f"Valores distintos (incluyendo NaN): {n}")
        print(df[col].value_counts(dropna=False).head(10).to_string())

        if n <= 1:
            constantes.append(col)
        else:
            con_varianza.append(col)

    print(f"Constantes: {constantes}")
    print(f"Con varianza:  {con_varianza}")

    return constantes, con_varianza

In [ ]:
constantes, con_varianza = inspeccionar_candidatos(df_FASE_valid)

El único atributo que si tiene distintos valores es TimeUnit: 
Esta asigna valores de unidad temporal a cada Fase, es decir, con que unidad de tiempo se miden los valores de tiempo (preparación y ejecución)
- 1 = Minutos
- 2 = Horas
- 3 = Dias
- 4 = Semanas
- 5 = Meses

Tenemos 273 registros en horas y 63568 en minutos, pasaremos de horas a minutos para tener todo el dataset en la misma escala.    
    (2) Horas --> (1) Minutos   


In [ ]:
#display(df_FASE[df_FASE['TimeUnit'] == 2]) # Fases con tiempo en dias

Tanto el tiempo de preparación como el tiempo de ejecución debo pasarlo a minutos para aquellos que tengan el valor `2` en la columna `TimeUnit `.  
        
Inspeccionado esas fases y hablando con oficina técnica vemos que realmente el error esta en la unidad especificada, pero los valores numéricos del tiempo si són correctos, con lo que no debemos modificar esos tiempos. Entonces podemos prescindir de la columna `TimeUnit` y tomar los tiempos como minitos.   

In [ ]:
def eliminar_columnas(df):
    """Elimina de df_FASES las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original)."""

    cols_sobran = [
        'CodMOTask','Notes','Order','RepetitionTime','DelimitationType', 'TimeUnit'
        'DelimitationDate','AllowFraccionate','MinimumFraction','RepetitionLot',
        'IDArticleExternal','RepetitionTimeReal','Group','SeriesRequired','QualityStatus',
        'InspectedDate','Priority','OptimizationFeature','Fixed','WithoutTimetable',
        'OccupancyType','DelimitationObligatory','IDBudgetCategory','ResourcePlanType',
        'SplitNumber','ManualPlanning','IDBudgetEmployee','IDBudgetMachine','IDBudgetTool',
        'IDBudgetMaterial','IDBudgetOtherCost','RowTimestamp','LastModifiedBy', 'WorkUnits',
        'IDProductFormula','LotsOrder','PrintLabels','AutoExecutionImputation',
        'IDSiteMaterial','IDWarehouseMaterial','GroupingNumber','CreationTimestamp',
        'RowCreatedBy','BlockedMaterial','BlockedCost','ImputationCostUpdated',
        'IDCheckList','NotOverlapCode','TiedCode','GroupedManufacturing','Blocked',
        'ReleaseKeptResources','DelayMargin','BlockedChangeMaterial','IDHourType','Color',
        'SetToQtyConversionFactor','AdjustTaskToDay','PercentProgressType','StatusTaskMES',
    ]

    cols_a_quitar = [c for c in cols_sobran if c in df_FASE.columns]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_FASE.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [ ]:
df_FASE = eliminar_columnas(df_FASE_sept)

display(df_FASE.head(1))

Perfecto, hasta aqui tengo df:
- sin las fases externas o canceladas
- sin fases con tiempos teoricos a 0
- instancias hechas en un intervalo de tiempo correcto
- sin los atributos que no aportan nada

## Eliminar fases que su OF debe finalizarse en Agosto
De esta forma solo contaremos con OF termindas, teniendo estas todo su tiempo imputado, siendo así los tiempos reales lícitos.  

In [5]:
df_OF = pd.read_parquet("res//procesed/df_OF_validas.parquet")
display(df_OF.head(1))
print(f"Total de OFs: {df_OF.shape[0]}")

,IDManufacturingOrder,CodCompany,CodManufacturingOrder,IDArticle,IDStructure,Description,TimeUnitDefault,IDStructureType,Notes,Quantity,...,LastModifiedBy,QuantityBaseFormula,IDUnitBaseFormula,IDArticleVersion,CreationTimestamp,RowCreatedBy,IDMaintenanceOrder,IDOrderMaterialPrevision,IDLaunchedMRP,AutoBatchAssignSalesOrders
0,000744d2-5b9e-40be-9126-9a8a66954f33,100,OF2502249,921ac292-f882-47f8-ae0b-ce82c4f1a3c8,9e856d02-3287-4b00-b968-3e6625ce343a,CONDUCTEUR 103127410,2,d3bd7234-a0af-4c35-b658-dcd8ac943c67,<p>A1</p>,7.0,...,JMAU,7.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb,NaN,2025-03-05 10:50:29.903,ICRE,NaN,NaN,622d3b9e-0ddb-4f25-9cee-4a75a8043531,0


Total de OFs: 19575
